In [3]:
"""
Save Pruned Models — ResNet-50 / CIFAR-10
==========================================
Reads pruning_benchmark_results.json, re-applies each method's best
configuration, and saves a .pth file per method under built_models/.

Output layout:
  built_models/
    baseline.pth
    unstructured_pruning.pth
    structured_pruning.pth
    magnitude_pruning.pth
    movement_pruning.pth
    lottery_ticket.pth
    iterative_pruning.pth
    oneshot_pruning.pth

Usage:
  python save_pruned_models.py \
    --baseline  ../__2__baseline_resnet50_cifar10.pth \
    --json      pruning_benchmark_results.json \
    --out       built_models
"""

import os, json, copy, warnings
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

# ── Constants ──────────────────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10
CIFAR_MEAN  = (0.4914, 0.4822, 0.4465)
CIFAR_STD   = (0.2023, 0.1994, 0.2010)

CALIBRATION_BATCHES  = 10   # movement pruning
ITERATIVE_STEP       = 0.15 # iterative pruning — must match benchmark


# ══════════════════════════════════════════════════════════════════════════════
# MODEL / DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def build_model():
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def load_baseline(path):
    model = build_model().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

def get_loader():
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def save(model, path):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1024**2
    sp = real_sparsity(model)
    print(f"    ✓  saved  {path}  ({mb:.1f} MB, sparsity={sp*100:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════════
# PRUNING IMPLEMENTATIONS  (mirror exactly what the benchmark does)
# ══════════════════════════════════════════════════════════════════════════════

# 1 ── Unstructured (global L1) ───────────────────────────────────────────────
def prune_unstructured(model, cfg):
    sparsity = cfg["target_sparsity"]
    pruned   = copy.deepcopy(model)
    layers   = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured,
                              amount=sparsity)
    for m, p in layers:
        prune.remove(m, p)
    return pruned


# 2 ── Structured (L1 filter) ─────────────────────────────────────────────────
def prune_structured(model, cfg):
    ratio  = cfg["filter_pruning_ratio"]
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, nn.Conv2d) and module.weight.shape[0] > 1:
            n = max(1, min(int(module.weight.shape[0] * ratio),
                           module.weight.shape[0] - 1))
            prune.ln_structured(module, name="weight", amount=n, n=1, dim=0)
            prune.remove(module, "weight")
    return pruned


# 3 ── Magnitude pruning ───────────────────────────────────────────────────────
def _mag_local_l1(model, sparsity):
    pruned = copy.deepcopy(model)
    for m, p in prunable_layers(pruned):
        prune.l1_unstructured(m, name=p, amount=sparsity)
        prune.remove(m, p)
    return pruned

def _mag_local_l2(model, sparsity):
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w    = module.weight.data
            flat = w.abs().pow(2).view(-1)
            thr  = torch.kthvalue(flat, max(1, int(flat.numel() * sparsity))).values
            module.weight.data = w * (w.abs().pow(2) > thr).float()
    return pruned

def _mag_global_l1(model, sparsity):
    pruned  = copy.deepcopy(model)
    layers  = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured,
                              amount=sparsity)
    for m, p in layers:
        prune.remove(m, p)
    return pruned

_MAG_FNS = {
    "local_l1":  _mag_local_l1,
    "local_l2":  _mag_local_l2,
    "global_l1": _mag_global_l1,
}

def prune_magnitude(model, cfg):
    fn = _MAG_FNS[cfg["criterion"]]
    return fn(model, cfg["sparsity"])


# 4 ── Movement pruning (Taylor |w*grad|) ─────────────────────────────────────
def _taylor_scores(model, loader, n_batches):
    model.train()
    criterion = nn.CrossEntropyLoss()
    model.zero_grad()
    for i, (inputs, targets) in enumerate(loader):
        if i >= n_batches: break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        criterion(model(inputs), targets).backward()
    scores = {}
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and module.weight.grad is not None:
            scores[name] = (module.weight.data * module.weight.grad
                            / n_batches).abs().detach().cpu()
    model.zero_grad(); model.eval()
    return scores

def prune_movement(model, cfg, loader):
    sparsity = cfg["target_sparsity"]
    pruned   = copy.deepcopy(model)
    scores   = _taylor_scores(pruned, loader, CALIBRATION_BATCHES)
    if not scores:
        layers = prunable_layers(pruned)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured,
                                  amount=sparsity)
        for m, p in layers: prune.remove(m, p)
        return pruned
    all_s = torch.cat([s.view(-1) for s in scores.values()])
    thr   = torch.kthvalue(all_s,
                           max(1, int(all_s.numel() * sparsity))).values.item()
    with torch.no_grad():
        for name, module in pruned.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in scores:
                module.weight.data *= (scores[name].to(DEVICE) > thr).float()
    return pruned


# 5 ── Lottery ticket ──────────────────────────────────────────────────────────
def prune_lottery_ticket(model, cfg):
    target_sp = cfg["target_sparsity"]
    n_rounds  = cfg["iterative_rounds"]
    per_round = 1.0 - (1.0 - target_sp) ** (1.0 / n_rounds)

    ticket = copy.deepcopy(model)
    for _ in range(n_rounds):
        layers = prunable_layers(ticket)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured,
                                  amount=per_round)
        for m, p in layers: prune.remove(m, p)

    mask = {name: (m.weight.data != 0).float()
            for name, m in ticket.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))}

    winning = copy.deepcopy(model)
    for name, module in winning.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in mask:
            module.weight.data *= mask[name].to(DEVICE)
    return winning


# 6 ── Iterative pruning ───────────────────────────────────────────────────────
def prune_iterative(model, cfg):
    target_sp = cfg["cumulative_sparsity"]
    current   = copy.deepcopy(model)

    while real_sparsity(current) < target_sp - 0.01:
        all_w = torch.cat([
            m.weight.data[m.weight.data != 0].abs().view(-1)
            for _, m in current.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))
        ])
        if all_w.numel() == 0: break
        k   = max(1, int(all_w.numel() * ITERATIVE_STEP))
        thr = torch.kthvalue(all_w, k).values.item()
        with torch.no_grad():
            for _, module in current.named_modules():
                if isinstance(module, (nn.Conv2d, nn.Linear)):
                    module.weight.data *= (module.weight.data.abs() > thr).float()

    return current


# 7 ── One-shot pruning ────────────────────────────────────────────────────────
class _L2Unstructured(prune.BasePruningMethod):
    PRUNING_TYPE = "unstructured"
    def compute_mask(self, t, default_mask):
        n       = t.nelement()
        n_prune = max(1, round(n * self.amount))
        mask    = default_mask.clone(memory_format=torch.contiguous_format)
        topk    = torch.topk(t.pow(2).view(-1), k=n - n_prune, largest=True)
        mask.view(-1).fill_(0)
        mask.view(-1)[topk.indices] = 1
        return mask
    @classmethod
    def apply(cls, module, name, amount):
        return super().apply(module, name, amount=amount)

def _oneshot_l1(model, s):
    p = copy.deepcopy(model)
    layers = prunable_layers(p)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=s)
    for mod, param in layers: prune.remove(mod, param)
    return p

def _oneshot_l2(model, s):
    p = copy.deepcopy(model)
    for mod, param in prunable_layers(p):
        _L2Unstructured.apply(mod, param, amount=s)
        prune.remove(mod, param)
    return p

def _oneshot_rand(model, s):
    p = copy.deepcopy(model)
    layers = prunable_layers(p)
    prune.global_unstructured(layers, pruning_method=prune.RandomUnstructured, amount=s)
    for mod, param in layers: prune.remove(mod, param)
    return p

_ONESHOT_FNS = {
    "l1_global": _oneshot_l1,
    "l2_global": _oneshot_l2,
    "random":    _oneshot_rand,
}

def prune_oneshot(model, cfg):
    fn = _ONESHOT_FNS[cfg["variant"]]
    return fn(model, cfg["sparsity"])


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    class args:
        baseline = "../__2__baseline_resnet50_cifar10.pth"
        json     = "pruning_benchmark_results.json"
        out      = "built_models"

    os.makedirs(args.out, exist_ok=True)

    print(f"\n  Loading JSON  : {args.json}")
    with open(args.json) as f:
        results = json.load(f)

    print(f"  Loading model : {args.baseline}")
    model  = load_baseline(args.baseline)
    loader = get_loader()        # needed only for movement pruning

    print(f"  Output dir    : {args.out}/\n")

    # ── baseline ──────────────────────────────────────────────────────────────
    print("  [0] baseline")
    save(model, os.path.join(args.out, "baseline.pth"))

    dispatch = {
        "unstructured_pruning": lambda cfg: prune_unstructured(model, cfg),
        "structured_pruning":   lambda cfg: prune_structured(model, cfg),
        "magnitude_pruning":    lambda cfg: prune_magnitude(model, cfg),
        "movement_pruning":     lambda cfg: prune_movement(model, cfg, loader),
        "lottery_ticket":       lambda cfg: prune_lottery_ticket(model, cfg),
        "iterative_pruning":    lambda cfg: prune_iterative(model, cfg),
        "oneshot_pruning":      lambda cfg: prune_oneshot(model, cfg),
    }

    labels = {
        "unstructured_pruning": "[1/7] Unstructured Pruning",
        "structured_pruning":   "[2/7] Structured Pruning",
        "magnitude_pruning":    "[3/7] Magnitude Pruning",
        "movement_pruning":     "[4/7] Movement Pruning",
        "lottery_ticket":       "[5/7] Lottery Ticket",
        "iterative_pruning":    "[6/7] Iterative Pruning",
        "oneshot_pruning":      "[7/7] One-Shot Pruning",
    }

    for key, fn in dispatch.items():
        if key not in results:
            print(f"  WARNING: '{key}' not found in JSON — skipping")
            continue

        cfg = results[key]["config_detail"]
        print(f"\n  {labels[key]}")
        print(f"    config : {results[key].get('best_config', cfg)}")

        pruned    = fn(cfg)
        out_path  = os.path.join(args.out, f"{key}.pth")
        save(pruned, out_path)

    print(f"\n  Done — all models saved to '{args.out}/'")
    print("  Files:")
    for f in sorted(os.listdir(args.out)):
        mb = os.path.getsize(os.path.join(args.out, f)) / 1024**2
        print(f"    {f:<35} {mb:6.1f} MB")


if __name__ == "__main__":
    main()


  Loading JSON  : pruning_benchmark_results.json
  Loading model : ../__2__baseline_resnet50_cifar10.pth
  Output dir    : built_models/

  [0] baseline
    ✓  saved  built_models\baseline.pth  (90.0 MB, sparsity=0.0%)

  [1/7] Unstructured Pruning
    config : sparsity=30%
    ✓  saved  built_models\unstructured_pruning.pth  (90.0 MB, sparsity=30.0%)

  [2/7] Structured Pruning
    config : filter_ratio=10%
    ✓  saved  built_models\structured_pruning.pth  (90.0 MB, sparsity=9.9%)

  [3/7] Magnitude Pruning
    config : global_l1 @ 30%
    ✓  saved  built_models\magnitude_pruning.pth  (90.0 MB, sparsity=30.0%)

  [4/7] Movement Pruning
    config : sparsity=30%
    ✓  saved  built_models\movement_pruning.pth  (90.0 MB, sparsity=32.1%)

  [5/7] Lottery Ticket
    config : sparsity=30%
    ✓  saved  built_models\lottery_ticket.pth  (90.0 MB, sparsity=6.9%)

  [6/7] Iterative Pruning
    config : sparsity=62.3%
    ✓  saved  built_models\iterative_pruning.pth  (90.0 MB, sparsity=62.3%)